# Q4: official held-out 500, base / A / B with label_score_v1

Unexecuted evaluation launcher. No training occurs. Upload the standalone `q4_heldout_eval.py` next to this notebook in `My Drive/AI_PUBMED/runs/qlora/optimized/`. The script embeds the exact frozen prompt and scorer, so no other helper upload is required.

Use a Colab T4 runtime. Base, A best_adapter and B best_adapter share NF4/double quantization, float16 compute/autocast, full contexts, non-thinking prompt rendering and the same one-forward label scorer. Training/dev/artificial overlap is checked before model loading. Gold labels are read only after all three models have 500 predictions. No generated JSON/evidence metrics are claimed.

The final file is `runs/qlora/optimized/q4_heldout_eval.json` on Drive. Ollama 65.4% / 0.469 remains a separate runtime and generation-protocol reference row. Existing training notebooks, checkpoints and adapters are read-only.


## 1. Install the pinned evaluation stack

This changes only the Colab session. Colab's CUDA PyTorch is retained. Restart this new runtime if requested, then continue from cell 2. The loader uses `dtype`, not the deprecated model-loading keyword. Upstream bitsandbytes warnings may still appear; its installed code is not patched.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.57.3", "peft==0.18.0", "bitsandbytes==0.48.2",
    "accelerate==1.12.0", "scikit-learn==1.7.2", "pyarrow==26.0.0"], check=True)


## 2. Mount Drive and verify paths

Adjust the three input paths if your folders differ. The output path is fixed under the project runs/qlora/optimized folder. Upload the Python script there first. It reads the existing data/ files and only the frozen best_adapter folders; it does not reselect checkpoints.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/AI_PUBMED")
SCRIPT = PROJECT / "runs/qlora/optimized/q4_heldout_eval.py"
RUN_A = PROJECT / "qlora_runs/pubmedqa_qlora_v1/A_train_only"
RUN_B = PROJECT / "runs/qlora/optimized/pubmedqa_qlora_label_score_v1/B_with_artificial"
assert SCRIPT.is_file(), "Upload q4_heldout_eval.py first."
for run in (RUN_A, RUN_B):
    assert (run / "summary.json").is_file(), run
    assert (run / "best_adapter/adapter_model.safetensors").is_file(), run
print("Result:", PROJECT / "runs/qlora/optimized/q4_heldout_eval.json")


## 3. Optional CPU precheck in Colab

This verifies the prompt fingerprint, all ID overlaps and token lengths without loading model weights or reading held-out gold labels. It writes q4_precheck.json and per-PMID q4_token_lengths.jsonl. An overlap aborts the command and reports the matching IDs.


In [ ]:
subprocess.run([sys.executable, "-u", str(SCRIPT), "--project-root", str(PROJECT), "--precheck-only"], check=True)


## 4. Run the three matched held-out evaluations

This cell starts GPU inference only. Any held-out overlap stops before model loading and is reported in q4_heldout_eval.json without scores. Predictions are cached per completed model under q4_heldout_eval_artifacts; reruns require the same frozen runtime/input/adapter manifest. No partial metric is reported. Completed final results are protected from overwrite.

The output includes accuracy, macro-F1, 3x3 confusion matrices (gold yes/no/maybe rows), selection provenance and the separate Ollama reference row. A was selected under its original generation metric; B under label_score_v1. These selection choices remain frozen.


In [ ]:
subprocess.run([sys.executable, "-u", str(SCRIPT), "--project-root", str(PROJECT),
    "--run-a-dir", str(RUN_A), "--run-b-dir", str(RUN_B)], check=True)
import json
result = json.loads((PROJECT / "runs/qlora/optimized/q4_heldout_eval.json").read_text("utf-8"))
for name, metrics in result["models"].items():
    print(name, "accuracy:", metrics["accuracy"], "macro-F1:", metrics["macro_f1"])
    print(metrics["confusion_matrix"])
print("Separate Ollama reference:", result["reference_rows"])
